In [1]:
import numpy as np
import pandas as pd 
import pickle

from pathlib import Path
from astropy.io import fits
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle
from scipy.stats import skew, kurtosis, shapiro
from utils.preprocessing import fourier_features, stetson_K, fourier_fit

In [2]:
# Root directory containing TESS FITS files
data_root = Path("/Users/trevin/Git/ucsd-phys-139-final/data")

out_pkl = data_root / "time_flux_pdcsap.pkl"
if True:
    print(f"{out_pkl} not found; generating from FITS files under {data_root}...")

    fits_files = sorted(data_root.rglob("*_lc.fits"))
    print(f"Found {len(fits_files)} light-curve FITS files under {data_root}")

    time_flux_list = []
    for fp in fits_files:
        try:
            with fits.open(fp, memmap=True) as hdul_all:
                lc = hdul_all[1].data
                t = np.array(lc['TIME'], dtype=float)
                f = np.array(lc['PDCSAP_FLUX'], dtype=float)

                # Try to get TICID from header; fall back to filename pattern
                ticid = None
                try:
                    ticid = hdul_all[0].header.get('TICID')
                    if ticid is None:
                        ticid = hdul_all[1].header.get('TICID')
                except Exception:
                    ticid = None

                if ticid is None:
                    parts = fp.name.split('-')
                    ticid = parts[2] if len(parts) > 2 else fp.stem

                star_id = str(ticid)
                time_flux_list.append((star_id, t, f))
        except Exception as e:
            print(f"Skipping {fp}: {e}")

    out_pkl.parent.mkdir(parents=True, exist_ok=True)
    with out_pkl.open("wb") as f:
        pickle.dump(time_flux_list, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved {len(time_flux_list)} time/flux pairs to {out_pkl}")
else:
    print(f"Using existing pickle: {out_pkl}")



/Users/trevin/Git/ucsd-phys-139-final/data/time_flux_pdcsap.pkl not found; generating from FITS files under /Users/trevin/Git/ucsd-phys-139-final/data...
Found 8943 light-curve FITS files under /Users/trevin/Git/ucsd-phys-139-final/data
Saved 8943 time/flux pairs to /Users/trevin/Git/ucsd-phys-139-final/data/time_flux_pdcsap.pkl


In [3]:
def compute_features_to_csv(light_curves, outfile="features.csv"):
    """
    light_curves: list of (t, f) tuples OR (star_id, t, f) triples
    outfile     : CSV file to write

    Returns the DataFrame of features.
    """

    rows = []

    for idx, item in enumerate(light_curves):
        # Support either (t, f) or (star_id, t, f)
        if isinstance(item, (list, tuple)) and len(item) == 3:
            star_id, t, f = item
        else:
            t, f = item
            star_id = idx 

        # remove NaNs
        mask = np.isfinite(t) & np.isfinite(f)
        t, f = t[mask], f[mask]

        # --- Period (Lomb–Scargle) ---
        freq, power = LombScargle(t, f).autopower()
        best_period = 1 / freq[np.argmax(power)]

        # --- Flux distribution features ---
        Q1 = np.percentile(f, 25)
        Q3 = np.percentile(f, 75)
        Q31 = Q3 - Q1
        Std = np.std(f)
        gamma1 = skew(f)
        gamma2 = kurtosis(f, fisher=True)
        W, _ = shapiro(f)
        K = stetson_K(f)

        # --- Fourier features ---
        R21, R31, phi21, phi31, Amp = fourier_features(best_period, t, f)

        # store (ensure star_id is first key)
        rows.append({
            "star_id": star_id,
            "period": best_period,
            "Q31": Q31,
            "Amp": Amp,
            "W": W,
            "K": K,
            "Std": Std,
            "gamma1": gamma1,
            "gamma2": gamma2,
            "R21": R21,
            "R31": R31,
            "phi21": phi21,
            "phi31": phi31
        })

        print(f"Processed light curve {idx+1}/{len(light_curves)}")

    df = pd.DataFrame(rows)
    # Reorder columns explicitly to keep star_id first
    ordered_cols = [
        "star_id", "period", "Q31", "Amp", "W", "K", "Std",
        "gamma1", "gamma2", "R21", "R31", "phi21", "phi31"
    ]
    df = df[ordered_cols]
    df.to_csv(outfile, index=False)
    print(f"\nSaved feature table to: {outfile}")

    return df

In [4]:
# Load list of (time, flux) tuples
pkl_path = Path("/Users/trevin/Git/ucsd-phys-139-final/data/time_flux_pdcsap.pkl")
if not pkl_path.exists():
    raise FileNotFoundError(f"Pickle not found: {pkl_path}")

with pkl_path.open("rb") as f:
    light_curves = pickle.load(f)

print(f"Loaded {len(light_curves)} light curves from {pkl_path}")

# Compute features and save to batched CSVs next to the pickle
output_dir = pkl_path.parent
output_dir.mkdir(parents=True, exist_ok=True)

batch_size = 500  # adjust as needed
n = len(light_curves)
print(f"Computing features in batches of {batch_size} (total {n})")

for start in range(0, n, batch_size):
    end = min(start + batch_size, n)
    batch = light_curves[start:end]
    batch_path = output_dir / f"features_batch_{start:05d}_{end:05d}.csv"

    # Skip already-processed batches
    if batch_path.exists():
        print(f"Skipping existing batch {start}:{end} -> {batch_path}")
        continue

    try:
        print(f"Processing batch {start}:{end} -> {batch_path}")
        df_batch = compute_features_to_csv(batch, outfile=str(batch_path))
        print(f"Saved {len(df_batch)} rows to {batch_path}")
        df = df_batch  # keep latest batch in memory if needed later
    except Exception as e:
        print(f"Error in batch {start}:{end}: {e}")
        # Continue with the next batch instead of failing the whole run
        continue

print("Finished processing all batches.")


Loaded 8943 light curves from /Users/trevin/Git/ucsd-phys-139-final/data/time_flux_pdcsap.pkl
Computing features in batches of 500 (total 8943)
Processing batch 0:500 -> /Users/trevin/Git/ucsd-phys-139-final/data/features_batch_00000_00500.csv
Processed light curve 1/500
Processed light curve 2/500
Processed light curve 3/500


/opt/homebrew/Caskroom/miniforge/base/envs/main/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 14499.
  res = hypotest_fun_out(*samples, **kwds)
/opt/homebrew/Caskroom/miniforge/base/envs/main/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 14732.
  res = hypotest_fun_out(*samples, **kwds)
/opt/homebrew/Caskroom/miniforge/base/envs/main/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 14891.
  res = hypotest_fun_out(*samples, **kwds)


Processed light curve 4/500
Processed light curve 5/500
Processed light curve 6/500
Processed light curve 7/500


/opt/homebrew/Caskroom/miniforge/base/envs/main/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 14852.
  res = hypotest_fun_out(*samples, **kwds)


Processed light curve 8/500
Processed light curve 9/500
Processed light curve 10/500


...

/opt/homebrew/Caskroom/miniforge/base/envs/main/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 13436.
  res = hypotest_fun_out(*samples, **kwds)


Processed light curve 436/443
Processed light curve 437/443
Processed light curve 438/443
Processed light curve 439/443


/opt/homebrew/Caskroom/miniforge/base/envs/main/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 13235.
  res = hypotest_fun_out(*samples, **kwds)
/opt/homebrew/Caskroom/miniforge/base/envs/main/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 12066.
  res = hypotest_fun_out(*samples, **kwds)


Processed light curve 440/443
Processed light curve 441/443
Processed light curve 442/443
Processed light curve 443/443

Saved feature table to: /Users/trevin/Git/ucsd-phys-139-final/data/features_batch_08500_08943.csv
Saved 443 rows to /Users/trevin/Git/ucsd-phys-139-final/data/features_batch_08500_08943.csv
Finished processing all batches.


/opt/homebrew/Caskroom/miniforge/base/envs/main/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 13508.
  res = hypotest_fun_out(*samples, **kwds)
